# 2차 — XGBoost rank:pairwise 손실함수 실험

## 배경
- LightGBM lambdarank는 group_size를 50~1000까지 바꿔봐도 전부 binary 기준(0.7400~0.74053)보다 낮았고,
  group_size가 커질수록 오히려 단조 감소(50→0.73748, 1000→0.72860) → 인공 그룹 기반 순위손실 자체가 이
  데이터셋엔 안 맞을 가능성이 높음
- 그래도 구현(부스팅 라이브러리)이 다르면 다를 수 있으니 XGBoost `rank:pairwise`로 같은 방식을 한 번 더 확인
- **같은 group_size 그리드로 비교**해서 lambdarank와 직접 대조 가능하게 구성

## 핵심 차이 (vs LightGBM lambdarank)
- group 구성 방식(셔플 후 고정 크기로 청크)은 동일
- `rank:pairwise`는 label_gain 같은 graded relevance 개념이 없음 (순수 0/1 선호 비교라 더 단순)
- XGBoost native API는 `DMatrix.set_group(group_sizes)`로 그룹 지정 (LightGBM의 group 파라미터와 같은 역할)
- eval_metric을 'auc'로 명시해서 objective와 무관하게 실제 평가지표를 직접 모니터링

## 전처리 / 파라미터 출처
- 전처리는 lambdarank 노트북과 완전히 동일 (TARGET_COL="임신 성공 여부", DEAD_COLS 2개 드롭,
  `is_DI`/`froze_embryo` 파생, 결측치 처리, LabelEncoder — train+test 합쳐서 fit)
- 시작 파라미터는 기존 `xgboost_best_params.json` (sklearn API 이름이지만 XGBoost가 거의 그대로 native
  파라미터로 인식: subsample/colsample_bytree는 이름 그대로, reg_alpha→alpha, reg_lambda→lambda alias)

---
**저장 위치:** `experiment_history/2차/2차_xgb_rankpairwise.ipynb` (같은 2차 폴더)


In [6]:
import json
import time
from pathlib import Path

import numpy as np
import pandas as pd
import xgboost as xgb
from scipy.stats import rankdata
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import roc_auc_score


In [7]:
# 노트북 위치: experiment_history/2차/
DATA_DIR = Path("../../data")
SHARED_DIR = Path("..")
SUBMIT_DIR = Path("../submit file")

TRAIN_PATH = DATA_DIR / "train.csv"
TEST_PATH = DATA_DIR / "test.csv"
TARGET_COL = "임신 성공 여부"
DEAD_COLS = ["불임 원인 - 여성 요인", "난자 채취 경과일"]

XGB_PARAMS_PATH = SHARED_DIR / "xgboost_best_params.json"

N_SPLITS = 5
SEED = 42
EARLY_STOPPING_ROUNDS = 100
N_BOOST_ROUND = 3000

# lambdarank에서 group_size가 작을수록 좋았던 패턴을 그대로 검증 — 동일 그리드 사용
CANDIDATE_GROUP_SIZES = [50, 100, 300, 1000]


## 1. 데이터 로드 + 전처리 (lambdarank 노트북과 동일)

In [8]:
train_raw = pd.read_csv(TRAIN_PATH).drop(columns=["ID"])
test_raw_full = pd.read_csv(TEST_PATH)
test_ids = test_raw_full["ID"]
test_raw = test_raw_full.drop(columns=["ID"])


def base_features(df):
    df = df.copy()
    df["is_DI"] = (df["시술 유형"] == "DI").astype(int)
    df["froze_embryo"] = df["동결 배아 사용 여부"].fillna(0).astype(int)
    return df


def fill_na(df):
    cat_cols = df.select_dtypes(include=["object"]).columns
    num_cols = df.select_dtypes(include=["int64", "float64"]).columns
    na_cols = df.isna().sum().loc[lambda x: x > 0].index
    for col in na_cols:
        if col in cat_cols:
            df[col] = df[col].fillna("해당없음")
        elif col in num_cols:
            df[col] = df[col].fillna(-1)
    return df


df_train = fill_na(base_features(train_raw.copy()).drop(columns=DEAD_COLS))
df_test = fill_na(base_features(test_raw.copy()).drop(columns=DEAD_COLS))

cat_cols = df_train.select_dtypes(include=["object"]).columns.tolist()

df_train_le = df_train.copy()
df_test_le = df_test.copy()
for col in cat_cols:
    le = LabelEncoder()
    combined = pd.concat([df_train_le[col], df_test_le[col]], axis=0).astype(str)
    le.fit(combined)
    df_train_le[col] = le.transform(df_train_le[col].astype(str))
    df_test_le[col] = le.transform(df_test_le[col].astype(str))

y = df_train_le[TARGET_COL].values.astype(np.float32)
X = df_train_le.drop(columns=[TARGET_COL])
X_test = df_test_le.copy()

print(f"전처리 완료: train {X.shape}, test {X_test.shape}")


/var/folders/s7/xbnkmtj92j5_ly_bxspy44f40000gn/T/ipykernel_6383/995283959.py:15: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = df.select_dtypes(include=["object"]).columns
/var/folders/s7/xbnkmtj92j5_ly_bxspy44f40000gn/T/ipykernel_6383/995283959.py:15: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://panda

전처리 완료: train (256351, 67), test (90067, 67)


## 2. CV 설정 + 시작 파라미터 로드

In [9]:
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
folds = list(skf.split(X, y))
print(f"{N_SPLITS}-fold 분할 완료 (seed={SEED})")


5-fold 분할 완료 (seed=42)


In [10]:
def load_base_params(path: Path) -> dict:
    try:
        with open(path, encoding="utf-8") as f:
            loaded = json.load(f)
        print(f"파라미터 출처: {path}")
        return loaded.get("best_params", loaded) if isinstance(loaded, dict) else loaded
    except FileNotFoundError:
        print(f"⚠ {path} 없음 — 기본 파라미터로 진행 (나중에 실제 튠 파일 경로/이름을 알려주시면 교체)")
        return {
            "max_depth": 5,
            "learning_rate": 0.02,
            "subsample": 0.8,
            "colsample_bytree": 0.8,
            "min_child_weight": 5,
            "reg_alpha": 1.0,
            "reg_lambda": 1.0,
        }

base_params = load_base_params(XGB_PARAMS_PATH)

# n_estimators는 num_boost_round(함수 인자)와 역할이 겹쳐서 분리
base_params.pop("n_estimators", None)
# binary objective 전용/레거시 키 제거
for k in ["objective", "eval_metric", "use_label_encoder", "scale_pos_weight"]:
    base_params.pop(k, None)

params = {
    **base_params,
    "objective": "rank:pairwise",
    "eval_metric": "auc",   # objective와 무관하게 실제 평가지표를 직접 모니터링 (auc는 높을수록 좋다고 자동 인식)
    "seed": SEED,
    "verbosity": 0,
}

print(json.dumps(params, indent=2, ensure_ascii=False))


⚠ ../xgboost_best_params.json 없음 — 기본 파라미터로 진행 (나중에 실제 튠 파일 경로/이름을 알려주시면 교체)
{
  "max_depth": 5,
  "learning_rate": 0.02,
  "subsample": 0.8,
  "colsample_bytree": 0.8,
  "min_child_weight": 5,
  "reg_alpha": 1.0,
  "reg_lambda": 1.0,
  "objective": "rank:pairwise",
  "eval_metric": "auc",
  "seed": 42,
  "verbosity": 0
}


## 3. fold별 학습 함수 정의 (lambdarank와 동일한 group 구성 방식)

In [11]:
def rank_normalize(scores: np.ndarray) -> np.ndarray:
    return rankdata(scores) / len(scores)


def make_group_sizes(n: int, group_size: int) -> list:
    sizes = [group_size] * (n // group_size)
    remainder = n % group_size
    if remainder:
        sizes.append(remainder)
    return sizes


def shuffle_rows(X_part: pd.DataFrame, y_part: np.ndarray, seed: int):
    rng = np.random.RandomState(seed)
    order = rng.permutation(len(X_part))
    return X_part.iloc[order].reset_index(drop=True), y_part[order]


def run_xgb_rank_cv(group_size: int, verbose: bool = False):
    oof_ = np.zeros(len(X))
    test_fold_preds_ = []
    fold_aucs_ = []

    start = time.time()
    for fold, (tr_idx, val_idx) in enumerate(folds):
        X_tr, X_val = X.iloc[tr_idx], X.iloc[val_idx]
        y_tr, y_val = y[tr_idx], y[val_idx]

        X_tr_shuf, y_tr_shuf = shuffle_rows(X_tr, y_tr, seed=SEED + fold)
        X_val_shuf, y_val_shuf = shuffle_rows(X_val, y_val, seed=SEED + fold + 1000)

        train_group = make_group_sizes(len(X_tr_shuf), group_size)
        valid_group = make_group_sizes(len(X_val_shuf), group_size)

        dtrain = xgb.DMatrix(X_tr_shuf, label=y_tr_shuf)
        dtrain.set_group(train_group)
        dvalid = xgb.DMatrix(X_val_shuf, label=y_val_shuf)
        dvalid.set_group(valid_group)

        booster = xgb.train(
            params,
            dtrain,
            num_boost_round=N_BOOST_ROUND,
            evals=[(dvalid, "valid")],
            early_stopping_rounds=EARLY_STOPPING_ROUNDS,
            verbose_eval=100 if verbose else False,
        )

        best_iter = booster.best_iteration
        # 예측은 원래 순서 그대로의 X_val/X_test (DMatrix는 group 없이 생성해도 predict엔 문제 없음)
        val_raw = booster.predict(xgb.DMatrix(X_val), iteration_range=(0, best_iter + 1))
        fold_auc = roc_auc_score(y_val, val_raw)
        fold_aucs_.append(fold_auc)
        oof_[val_idx] = rank_normalize(val_raw)

        test_raw = booster.predict(xgb.DMatrix(X_test), iteration_range=(0, best_iter + 1))
        test_fold_preds_.append(rank_normalize(test_raw))

    elapsed_ = time.time() - start
    return oof_, test_fold_preds_, fold_aucs_, elapsed_


## 4. group_size 그리드서치

In [12]:
results = {}
print(f"{'group_size':>10} | {'OOF AUC':>8} | {'fold_std':>9} | {'time(s)':>8}")
print("-" * 45)
for gs in CANDIDATE_GROUP_SIZES:
    oof_gs, test_preds_gs, fold_aucs_gs, elapsed = run_xgb_rank_cv(gs)
    oof_auc_gs = roc_auc_score(y, oof_gs)
    fold_std_gs = float(np.std(fold_aucs_gs))
    results[gs] = {
        "oof": oof_gs,
        "test_fold_preds": test_preds_gs,
        "fold_aucs": fold_aucs_gs,
        "oof_auc": oof_auc_gs,
        "fold_std": fold_std_gs,
        "elapsed": elapsed,
    }
    print(f"{gs:>10} | {oof_auc_gs:>8.5f} | {fold_std_gs:>9.5f} | {elapsed:>8.1f}")

best_gs = max(results, key=lambda g: results[g]["oof_auc"])
print(f"\n채택: group_size={best_gs}  (OOF AUC={results[best_gs]['oof_auc']:.5f})")
print()
print("비교 기준:")
print("  기존 binary LightGBM 단일시드 OOF: 0.7400 ~ 0.74053")
print("  멀티시드 배깅 OOF: 0.74036 (기존) / 0.74037 (견고탐색)")
print("  팀 블렌딩 OOF: 0.74071")
print("  LightGBM lambdarank 최고치: 0.73748 (group_size=50)")


group_size |  OOF AUC |  fold_std |  time(s)
---------------------------------------------
        50 |  0.73939 |   0.00138 |     49.2
       100 |  0.73781 |   0.00131 |     64.1
       300 |  0.73584 |   0.00126 |    101.6
      1000 |  0.73519 |   0.00148 |    144.5

채택: group_size=50  (OOF AUC=0.73939)

비교 기준:
  기존 binary LightGBM 단일시드 OOF: 0.7400 ~ 0.74053
  멀티시드 배깅 OOF: 0.74036 (기존) / 0.74037 (견고탐색)
  팀 블렌딩 OOF: 0.74071
  LightGBM lambdarank 최고치: 0.73748 (group_size=50)


## 5. 최적 group_size 결과로 확정

In [13]:
oof = results[best_gs]["oof"]
test_fold_preds = results[best_gs]["test_fold_preds"]
fold_aucs = results[best_gs]["fold_aucs"]
oof_auc = results[best_gs]["oof_auc"]

print(f"채택된 group_size={best_gs}")
print(f"Fold별 AUC: {[round(a, 5) for a in fold_aucs]}")
print(f"Fold AUC 평균±표준편차: {np.mean(fold_aucs):.5f} ± {np.std(fold_aucs):.5f}")
print(f"전체 OOF AUC (rank-정규화 기준): {oof_auc:.5f}")


채택된 group_size=50
Fold별 AUC: [0.73824, 0.74151, 0.73944, 0.73763, 0.74013]
Fold AUC 평균±표준편차: 0.73939 ± 0.00138
전체 OOF AUC (rank-정규화 기준): 0.73939


## 6. test 예측 평균 + 저장

In [14]:
test_pred = np.mean(test_fold_preds, axis=0)

BLEND_DIR = SHARED_DIR / "blend_cache"
BLEND_DIR.mkdir(exist_ok=True)
np.save(BLEND_DIR / "xgb_rankpairwise_oof.npy", oof)
np.save(BLEND_DIR / "xgb_rankpairwise_test.npy", test_pred)
print(f"저장 완료: {BLEND_DIR / 'xgb_rankpairwise_oof.npy'}, {BLEND_DIR / 'xgb_rankpairwise_test.npy'}")

SUBMIT_DIR.mkdir(exist_ok=True)
submission = pd.DataFrame({"ID": test_ids, "probability": test_pred})
out_path = SUBMIT_DIR / f"2차_xgb_rankpairwise_local{oof_auc:.5f}.csv"
submission.to_csv(out_path, index=False)
print(f"비제출용 결과 저장: {out_path}")


저장 완료: ../blend_cache/xgb_rankpairwise_oof.npy, ../blend_cache/xgb_rankpairwise_test.npy
비제출용 결과 저장: ../submit file/2차_xgb_rankpairwise_local0.73939.csv


## 7. 결과 해석

- **0.74037을 넘는 group_size가 있다면**: XGBoost 구현 특성상 LightGBM과 다르게 동작한 것 →
  블렌딩 후보로 검토.
- **lambdarank와 비슷한 패턴(작을수록 낫지만 여전히 기준 이하)이라면**: 인공 그룹 기반 순위손실 자체가
  이 데이터셋/전처리 조합에는 구조적으로 안 맞는다는 결론을 더 확실히 뒷받침 — binary objective +
  멀티시드 배깅이 여전히 최선.
